In [1]:
import pandas as pd
df = pd.read_csv("test.csv")
df.head(2)

,id,question,answer,answer_idx,meta_info,option_A,option_B,option_C,option_D,option_E,...,run2_initial_answer,run2_initial_confidence,run2_final_answer,run2_final_confidence,run3_answer,run3_confidence,run4_initial_answer,run4_initial_confidence,run4_final_answer,run4_final_confidence
0,Q0001,A junior orthopaedic surgery resident is compl...,Tell the attending that he cannot fail to disc...,C,step1,Disclose the error to the patient but leave it...,Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,Refuse to dictate the operative report,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q0002,A 67-year-old man with transitional cell carci...,Cross-linking of DNA,E,step1,Inhibition of thymidine synthesis,Inhibition of proteasome,Hyperstabilization of microtubules,Generation of free radicals,Cross-linking of DNA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Gemini 3.5 Flash RUN 2

In [7]:
import os
import json
import time
import threading
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel, Field
from typing import Literal
from google import genai
from google.genai import types


client = genai.Client()

INPUT_CSV = "test.csv"
OUTPUT_JSON = "run2_gemini3_5_flash_responses.json"
OUTPUT_CSV = "run2_gemini3_5_flash.csv"

PROCESS_N = 15
MAX_WORKERS = 1

save_lock = threading.Lock()

df = pd.read_csv(INPUT_CSV)
df_run = df if PROCESS_N is None else df.head(PROCESS_N)


class MCQAnswer(BaseModel):
  answer: Literal["A", "B", "C", "D", "E"]
  confidence: int = Field(..., ge=0, le=100)


config = types.GenerateContentConfig(
  response_mime_type="application/json",
  response_schema=MCQAnswer,
)


def build_prompt(row):
  return f"""{row["question"]}

A. {row["option_A"]}
B. {row["option_B"]}
C. {row["option_C"]}
D. {row["option_D"]}
E. {row["option_E"]}

Provide only the final answer choice (A/B/C/D/E).
Then provide a confidence score from 0 to 100%, defined as your estimated probability that the selected answer is correct.
"""


def save_outputs(outputs):
  with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)

  out_df = pd.DataFrame(outputs)

  expected_cols = [
    "id",
    "run2_initial_answer",
    "run2_initial_confidence",
    "run2_answer",
    "run2_confidence",
  ]

  for col in expected_cols:
    if col not in out_df.columns:
      out_df[col] = None

  out_df = out_df[expected_cols]
  out_df.to_csv(OUTPUT_CSV, index=False)


def call_gemini_with_retry(qid, prompt, max_attempts=1):
  for attempt in range(max_attempts):
    try:
      chat = client.chats.create(
        model="gemini-3.5-flash",
        config=config,
      )

      initial_response = chat.send_message(prompt)
      initial_parsed = json.loads(initial_response.text)

      followup_response = chat.send_message("Are you sure?")
      followup_parsed = json.loads(followup_response.text)

      return {
        "id": qid,
        "run2_initial_answer": initial_parsed["answer"],
        "run2_initial_confidence": int(initial_parsed["confidence"]),
        "run2_answer": followup_parsed["answer"],
        "run2_confidence": int(followup_parsed["confidence"]),
      }

    except Exception as e:
      print(f"Error on {qid}, attempt {attempt + 1}: {e}")
      time.sleep(2 * (attempt + 1))

  return {
    "id": qid,
    "run2_initial_answer": None,
    "run2_initial_confidence": None,
    "run2_answer": None,
    "run2_confidence": None,
  }


# Resume from CSV
if os.path.exists(OUTPUT_CSV):
  existing_df = pd.read_csv(OUTPUT_CSV)

  expected_cols = [
    "id",
    "run2_initial_answer",
    "run2_initial_confidence",
    "run2_answer",
    "run2_confidence",
  ]

  for col in expected_cols:
    if col not in existing_df.columns:
      existing_df[col] = None

  existing_df = existing_df[expected_cols]
  json_outputs = existing_df.to_dict(orient="records")

  completed_ids = set(existing_df["id"].astype(str))

  print(f"Resuming from {OUTPUT_CSV}")
  print(f"Already completed: {len(completed_ids)}")

else:
  json_outputs = []
  completed_ids = set()
  print("Starting from beginning")


tasks = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
  for _, row in df_run.iterrows():
    qid = str(row["id"])

    if qid in completed_ids:
      print(f"Skipping {qid}")
      continue

    prompt = build_prompt(row)
    future = executor.submit(call_gemini_with_retry, qid, prompt)
    tasks.append(future)

  for future in as_completed(tasks):
    item = future.result()

    with save_lock:
      json_outputs.append(item)
      save_outputs(json_outputs)

    print(
      f'{item["id"]}: {item["run2_initial_answer"]}, '
      f'{item["run2_initial_confidence"]}% | '
      f'{item["run2_answer"]}, '
      f'{item["run2_confidence"]}%'
    )
    print("Saved progress")


print("Done.")
print(f"Saved {OUTPUT_JSON}")
print(f"Saved {OUTPUT_CSV}")

Resuming from run2_gemini3_5_flash.csv
Already completed: 351
Skipping Q0001
Skipping Q0002
Skipping Q0003
Skipping Q0004
Skipping Q0005
Skipping Q0006
Skipping Q0007
Skipping Q0008
Skipping Q0009
Skipping Q0010
Skipping Q0011
Skipping Q0012
Skipping Q0013
Skipping Q0014
Error on Q0015, attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Q0015: None, None% | None, None%
Saved progress
Done.
Saved run2_gemini3_5_flash_responses.json
Saved run2_gemini3_5_flash.csv
